In [9]:
import os
from pathlib import Path
# Backward-compatible PDF loaders for langchain and langchain_community
try:
    from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
except ModuleNotFoundError:
    from langchain.document_loaders import PyPDFLoader, PyMuPDFLoader
try:
    from langchain.text_splitters import RecursiveCharacterTextSplitter
except Exception:
    try:
        from langchain.text_splitter import RecursiveCharacterTextSplitter
    except Exception:
        class RecursiveCharacterTextSplitter:
            def __init__(self, chunk_size=1000, chunk_overlap=200):
                self.chunk_size = chunk_size
                self.chunk_overlap = chunk_overlap
            def split_documents(self, documents):
                # simple fallback: return original documents as single chunk
                return documents

print("Imports OK")

C:\Users\Service pc\AppData\Local\Temp\ipykernel_13896\4246011191.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [10]:
### read a PDF file inside a directory
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error processing: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")        

Found 1 PDF files to process

Processing : Resume_Abhishek_raut.pdf
Loaded 1 pages

Total documents loaded: 1


In [11]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-24T07:13:05+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-24T07:13:05+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\Resume_Abhishek_raut.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume_Abhishek_raut.pdf', 'file_type': 'pdf'}, page_content='Abhishek Raut\nrautabhishek919@gmail.com|+91 8080967909|LinkedIn|GitHub|Portfolio\nObjective\nData Science and Machine Learning practitioner skilled in Python, SQL, and advanced ML/DL libraries. Experienced in exploratory\ndata analysis (EDA), predictive modeling, fraud detection, and building interactive BI dashboards. Dedicated to extracting actionable\ninsights from data and engineering scalable, production-ready analytical solutions.\nEducation\nGramaudyogik Sh

In [12]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    # Try several constructor signatures for compatibility across langchain versions
    splitter_args_candidates = [
        {"chunk_size": chunk_size, "chunk_overlap": chunk_overlap, "length_function": len, "separators": ["\n\n", "\n", " ", ""]},
        {"chunk_size": chunk_size, "chunk_overlap": chunk_overlap, "length_function": len},
        {"chunk_size": chunk_size, "chunk_overlap": chunk_overlap},
        {}
    ]
    text_splitter = None
    last_err = None
    for kwargs in splitter_args_candidates:
        try:
            text_splitter = RecursiveCharacterTextSplitter(**kwargs)
            if kwargs:
                print(f"Using splitter kwargs: {list(kwargs.keys())}")
            else:
                print("Using default splitter constructor")
            break
        except TypeError as e:
            last_err = e
            continue
    if text_splitter is None:
        print("Failed to create text splitter, returning original documents:", last_err)
        return documents

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        try:
            print(f"content: {split_docs[0].page_content[:200]}...")
            print(f"Metadata: {split_docs[0].metadata}")
        except Exception:
            print(str(split_docs[0])[:200])
    return split_docs
    

In [13]:
chunks=split_documents(all_pdf_documents)
chunks

Using splitter kwargs: ['chunk_size', 'chunk_overlap']
Split 1 documents into 1 chunks

Example chunk:
content: Abhishek Raut
rautabhishek919@gmail.com|+91 8080967909|LinkedIn|GitHub|Portfolio
Objective
Data Science and Machine Learning practitioner skilled in Python, SQL, and advanced ML/DL libraries. Experien...
Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-24T07:13:05+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-24T07:13:05+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\Resume_Abhishek_raut.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume_Abhishek_raut.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-24T07:13:05+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-24T07:13:05+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\Resume_Abhishek_raut.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume_Abhishek_raut.pdf', 'file_type': 'pdf'}, page_content='Abhishek Raut\nrautabhishek919@gmail.com|+91 8080967909|LinkedIn|GitHub|Portfolio\nObjective\nData Science and Machine Learning practitioner skilled in Python, SQL, and advanced ML/DL libraries. Experienced in exploratory\ndata analysis (EDA), predictive modeling, fraud detection, and building interactive BI dashboards. Dedicated to extracting actionable\ninsights from data and engineering scalable, production-ready analytical solutions.\nEducation\nGramaudyogik Sh

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()   
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1054.68it/s]


Loaded embedding model: all-MiniLM-L6-v2


In [16]:
class vectorstore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings"},
            )

            print(f"Vector store initialized at {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")

        print(f"Adding {len(documents)} documents to vector store...")
        ids = []
        metadatas = []
        documents_content = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}-{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_content.append(doc.page_content)
            embedding_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                embeddings=embedding_list,
                documents=documents_content,
            )
            print(f"Added {len(documents)} documents to vector store")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

store = vectorstore()
store

Vector store initialized at pdf_documents
Existing documents in collection: 5


In [17]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-06-24T07:13:05+00:00', 'author': '', 'keywords': '', 'moddate': '2026-06-24T07:13:05+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\Resume_Abhishek_raut.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Resume_Abhishek_raut.pdf', 'file_type': 'pdf'}, page_content='Abhishek Raut\nrautabhishek919@gmail.com|+91 8080967909|LinkedIn|GitHub|Portfolio\nObjective\nData Science and Machine Learning practitioner skilled in Python, SQL, and advanced ML/DL libraries. Experienced in exploratory\ndata analysis (EDA), predictive modeling, fraud detection, and building interactive BI dashboards. Dedicated to extracting actionable\ninsights from data and engineering scalable, production-ready analytical solutions.\nEducation\nGramaudyogik Sh

In [18]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

store.add_documents(chunks,embeddings)


Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Adding 1 documents to vector store...
Added 1 documents to vector store
Total documents in collection after addition: 6


In [19]:
class RAGRetriever:
    def __init__(self, vector_store: vectorstore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: {query}")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        if self.vector_store is None or getattr(self.vector_store, "collection", None) is None:
            print("No vector store collection available.")
            return []

        if getattr(self.vector_store.collection, "count", lambda: 0)() == 0:
            print("The vector store is empty.")
            return []

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            documents = results.get("documents") or []
            metadatas = results.get("metadatas") or []
            distances = results.get("distances") or []
            ids = results.get("ids") or []

            if not documents or not documents[0]:
                print("No documents retrieved from the vector store.")
                return []

            retrieved_docs = []
            docs = documents[0]
            meta_items = metadatas[0] if metadatas and len(metadatas) > 0 else []
            distance_items = distances[0] if distances and len(distances) > 0 else []
            id_items = ids[0] if ids and len(ids) > 0 else []

            for i, (doc_id, document_text, metadata, distance) in enumerate(zip(id_items, docs, meta_items, distance_items)):
                similarity_score = max(0.0, 1.0 - float(distance))
                print(f"Rank {i + 1}: distance={distance}, score={similarity_score}")
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document_text,
                        "metadata": metadata,
                        "score": similarity_score,
                        "distance": distance,
                        "rank": i + 1,
                    })

            print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(store, embedding_manager)
rag_retriever

In [20]:
rag_retriever

In [21]:
rag_retriever.retrieve("What is the main topic of the document?", top_k=5, score_threshold=0.0)

Retrieving documents for query: What is the main topic of the document?
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Rank 1: distance=1.8666805028915405, score=0.0
Rank 2: distance=1.8666805028915405, score=0.0
Rank 3: distance=1.8666805028915405, score=0.0
Rank 4: distance=1.8666805028915405, score=0.0
Rank 5: distance=1.8666805028915405, score=0.0
Retrieved 5 documents (after filtering)


[{'id': 'doc_dd8b6faf-0',
  'content': 'Abhishek Raut\nrautabhishek919@gmail.com|+91 8080967909|LinkedIn|GitHub|Portfolio\nObjective\nData Science and Machine Learning practitioner skilled in Python, SQL, and advanced ML/DL libraries. Experienced in exploratory\ndata analysis (EDA), predictive modeling, fraud detection, and building interactive BI dashboards. Dedicated to extracting actionable\ninsights from data and engineering scalable, production-ready analytical solutions.\nEducation\nGramaudyogik Shikshan Mandal Maharashtra Institute of T echnology (MIT)Chh. Sambhajinagar, India\nB.Tech in Artificial Intelligence & Data Science – CGPA: 7.63 2022 – 2026\nShreeram Junior College Beed, India\n12th Grade – Percentage: 73.83% 2021 – 2022\nExperience\nData Science Intern Jan 2026 – May 2026\nMain Flow Services & Technologies Pvt. Ltd.\n• Performed exploratory data analysis (EDA) on real-world datasets using Pandas, NumPy, Matplotlib, and Seaborn to uncover\npatterns, trends, and actiona

In [22]:
print('collection count:', store.collection.count())
print('sample docs:', store.collection.get(include=['documents','metadatas'])['documents'][:3])
query_embedding = embedding_manager.generate_embeddings(['What is the main topic of the document?'])[0]
results = store.collection.query(query_embeddings=[query_embedding.tolist()], n_results=5)
print(results)
print('documents keys:', results.keys())

collection count: 6
sample docs: ['Abhishek Raut\nrautabhishek919@gmail.com|+91 8080967909|LinkedIn|GitHub|Portfolio\nObjective\nData Science and Machine Learning practitioner skilled in Python, SQL, and advanced ML/DL libraries. Experienced in exploratory\ndata analysis (EDA), predictive modeling, fraud detection, and building interactive BI dashboards. Dedicated to extracting actionable\ninsights from data and engineering scalable, production-ready analytical solutions.\nEducation\nGramaudyogik Shikshan Mandal Maharashtra Institute of T echnology (MIT)Chh. Sambhajinagar, India\nB.Tech in Artificial Intelligence & Data Science – CGPA: 7.63 2022 – 2026\nShreeram Junior College Beed, India\n12th Grade – Percentage: 73.83% 2021 – 2022\nExperience\nData Science Intern Jan 2026 – May 2026\nMain Flow Services & Technologies Pvt. Ltd.\n• Performed exploratory data analysis (EDA) on real-world datasets using Pandas, NumPy, Matplotlib, and Seaborn to uncover\npatterns, trends, and actionable i

In [23]:
rag_retriever.retrieve("")

Retrieving documents for query: 
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Rank 1: distance=1.837303876876831, score=0.0
Rank 2: distance=1.837303876876831, score=0.0
Rank 3: distance=1.837303876876831, score=0.0
Rank 4: distance=1.837303876876831, score=0.0
Rank 5: distance=1.837303876876831, score=0.0
Retrieved 5 documents (after filtering)


[{'id': 'doc_dd8b6faf-0',
  'content': 'Abhishek Raut\nrautabhishek919@gmail.com|+91 8080967909|LinkedIn|GitHub|Portfolio\nObjective\nData Science and Machine Learning practitioner skilled in Python, SQL, and advanced ML/DL libraries. Experienced in exploratory\ndata analysis (EDA), predictive modeling, fraud detection, and building interactive BI dashboards. Dedicated to extracting actionable\ninsights from data and engineering scalable, production-ready analytical solutions.\nEducation\nGramaudyogik Shikshan Mandal Maharashtra Institute of T echnology (MIT)Chh. Sambhajinagar, India\nB.Tech in Artificial Intelligence & Data Science – CGPA: 7.63 2022 – 2026\nShreeram Junior College Beed, India\n12th Grade – Percentage: 73.83% 2021 – 2022\nExperience\nData Science Intern Jan 2026 – May 2026\nMain Flow Services & Technologies Pvt. Ltd.\n• Performed exploratory data analysis (EDA) on real-world datasets using Pandas, NumPy, Matplotlib, and Seaborn to uncover\npatterns, trends, and actiona

RAG Pipeline

In [24]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os
from pathlib import Path

env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)

# Read the key from the project .env file, supporting both GROQ_API_KEY and groq_api_key names.
groq_api_key = os.getenv("GROQ_API_KEY") or os.getenv("GROQ_APIKEY") or os.getenv("groq_api_key")
if not groq_api_key:
    raise ValueError("No Groq API key found. Add it to the project .env file or set GROQ_API_KEY.")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.7,
    max_tokens=1024,
)


def rag_qa(query: str, retriever, llm_instance, top_k: int = 3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    print(f"Context:\n{context}\n")
    if not context:
        return "No relevant documents found."

    prompt = f"""Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:"""

    response = llm_instance.invoke([prompt])
    return response.content

In [25]:
answer = rag_qa("what is attention Mechanism", rag_retriever, llm)
print(answer)

Retrieving documents for query: what is attention Mechanism
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Rank 1: distance=1.9666541814804077, score=0.0
Rank 2: distance=1.9666541814804077, score=0.0
Rank 3: distance=1.9666541814804077, score=0.0
Retrieved 3 documents (after filtering)
Context:
Abhishek Raut
rautabhishek919@gmail.com|+91 8080967909|LinkedIn|GitHub|Portfolio
Objective
Data Science and Machine Learning practitioner skilled in Python, SQL, and advanced ML/DL libraries. Experienced in exploratory
data analysis (EDA), predictive modeling, fraud detection, and building interactive BI dashboards. Dedicated to extracting actionable
insights from data and engineering scalable, production-ready analytical solutions.
Education
Gramaudyogik Shikshan Mandal Maharashtra Institute of T echnology (MIT)Chh. Sambhajinagar, India
B.Tech in Artificial Intelligence & Data Science – CGPA: 7.63 2022 – 2026
Shreeram Junior Colleg

Enhanced RAG Pipeline


In [26]:
def rag_advanced(query, retriever, llm, top=5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query, top_k=top, score_threshold=min_score)
    if not results:
        return {
            'answer': 'No relevant documents found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }
    context = "\n\n".join([doc["content"] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc.get('similarity_score', None),
        'preview': doc['content'][:120] + '...'
    } for doc in results]
    confidence = max([doc.get('similarity_score', 0.0) for doc in results])

    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

if 'rag_retriever' not in globals():
    rag_retriever = RAGRetriever(store, embedding_manager)

result = rag_advanced("Fraut Detection", rag_retriever, llm, top=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("sources:", result['sources'])
print("confidence:", result["confidence"])
print("context preview:", result['context'][:300])        



Retrieving documents for query: Fraut Detection
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Rank 1: distance=1.7939060926437378, score=0.0
Rank 2: distance=1.7939060926437378, score=0.0
Rank 3: distance=1.7939060926437378, score=0.0
Retrieved 0 documents (after filtering)
Answer: No relevant documents found.
sources: []
confidence: 0.0
context preview: 


In [27]:
from typing import List , Dict , Any
import time

class AdvancedRAGPipeline:
    def  __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self , question: str, top_k = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) ->Dict[str , any]:
        results = self.retriever.retrieve(question, top_k = top_k, score_threshold = min_score)
        if not results:
            answer = "no releveant context found"
            sources =[]
            context = ""
        else:
            context = "\n\n".join([doc['content']for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source','unknown')),
                'page' : doc['metadata'].get('page', 'unknown'),
                'score' : doc['content'][:120]+'...'
            } for doc in results]

            prompt = f"""use the following context to answer the question concisely.\ncontext:\n{context}\n\nQuestion:{question}"""
            if stream:
                print("Streaming answer")
                for i in range(0, len(prompt),80):
                    print(prompt[i:i+80],end='',flush = True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context = context , question = question)])
            answer = response.content

            citations = [f"[{i+1}]{src['source']}(page{src['page']})" for i, src in enumerate(sources)]
            answer_with_citations = answer + "\n\nCitations:\n"+"\n".join(citations) if citations else answer

            summary = None
            if summarize and answer:
                summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
                summary_resp = self.llm.invoke([summary_prompt])
                summary = summary_resp.content

                self.history.append({
                    'question': question,
                    'answer' : answer,
                    'sources' : sources,
                    'summary' : summary
                })

                return{
                    'question': question,
                    'answer' : answer_with_citations,
                    'sources': sources,
                    'summary' : summary,
                    'history' : self.history
                }


        adv_rag = AdvancedRAGPipeline(rag_retriever,llm)
        result = adv_rag.query("what is positional encoding?", top_k=3, min_score=0.3,stream=True, summarize=True)
        print("\nFinal Answer: ", result['asnwer'])
        print('Summary:',result['summary'])
        print("history", result['history'][-1])
